In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
g = torch.Generator().manual_seed(2147483647)

In [ ]:
words = open('../../assets/names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))

In [ ]:
# String Conversions
stoi = {c : i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i : c for c, i in stoi.items()}

In [ ]:
xs = []
ys = []

for w in words:
    chs = ['.'] + list(w) + ['.']

    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])


xs = torch.tensor(xs)
ys = torch.tensor(ys)

nums = xs.nelement()

print(f'Number of samples: {nums}')

In [ ]:
W = torch.randn(27, 27, requires_grad=True)

In [ ]:
for _ in range(500):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W # Logits
    counts = logits.exp()

    probs = counts / counts.sum(dim = 1, keepdim=True)
    loss = -probs[torch.arange(len(ys)), ys].log().mean() + 0.01 * ( W ** 2 ).mean()

    print(loss.item())

    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

In [ ]:
for i in range(5):
    ix = 0
    out = []
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()

        p = counts/counts.sum(dim=1, keepdim= True)

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        
        if type(ix) is int:
            out.append(itos[ix])

        if ix == 0:
            break
        
    print(''.join(out))